# TP3 — Agentic Patterns & Multi-Agent Systems with CrewAI
## Module 2IAiasd 9.7 — Agentic AI · IMT Mines Alès

**Duration:** ~1 hour (first half of the session — TP4 LangGraph follows)
**Prerequisites:**
- TP1 completed — you know the **ReAct loop** (reason → act → observe)
- TP2 completed — the `chroma_db/` folder (index of your 2A internship report) is in the same directory as this notebook
- `ollama serve` is running, `qwen2.5:3b` and `nomic-embed-text` are pulled

**Objective:** Go beyond ReAct. Implement two other agentic patterns from CM1 — **reflection** and **planning** —
by hand, then build a **multi-agent system** with CrewAI where a *researcher* and a *writer* collaborate
on your internship report.

---

### Structure

| # | Section | Concepts | Time |
|---|---------|----------|------|
| 1 | Reflection | generate → critique → revise, structured critic | 15 min |
| 2 | Planning | Plan-and-Execute, Tree of Thoughts (ToT) | 15 min |
| 3 | CrewAI | agents with role / goal / backstory, tasks, crew — researcher + writer | 30 min |

### What you will build today

```
  Section 1                Section 2                       Section 3
  ─────────                ─────────                       ─────────
  ┌─────────┐              goal ──► PLANNER ──► steps       inputs {topic}
  │GENERATOR│◄──┐                       │                        │
  └────┬────┘   │ revise                ▼                   ┌────▼──────┐  search_report
       ▼        │              EXECUTOR (step by step)      │RESEARCHER │◄── (ChromaDB TP2)
  ┌─────────┐   │                                           └────┬──────┘
  │ CRITIC  │───┘           problem ──► k ideas ──► scores       ▼ notes
  └─────────┘                               └──► best ──►   ┌───────────┐
                                                 develop    │  WRITER   │──► summary.md
                                                            └───────────┘
```

> 📖 **Reference:** CM1 — the four agentic patterns (reflection, tool use, planning, multi-agent).
> In TP1 you implemented *tool use* (ReAct). Today: the three others.

---
## Setup

In [1]:
%pip install -q crewai langchain-ollama langchain-chroma chromadb pydantic

print("✅ Dependencies installed")

Note: you may need to restart the kernel to use updated packages.
✅ Dependencies installed


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.57.0 requires pillow<13,>=7.1.0, but you have pillow 13.0.0 which is incompatible.

[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, textwrap, requests
from pathlib import Path
from pydantic import BaseModel, Field

# CrewAI: disable telemetry and the interactive "view traces?" prompt
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"]        = "true"
os.environ["CREWAI_TRACING_ENABLED"]   = "false"

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage

MODEL       = "qwen2.5:3b"
CREW_MODEL  = "qwen2.5:3b"   # 👉 use "qwen2.5:7b" if your laptop has ≥ 16 GB RAM (much more reliable with CrewAI)
EMBED_MODEL = "nomic-embed-text"
CHROMA_DIR  = "./chroma_db"  # built in TP2 Part 1

# Health check
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=3)
    models = [m["name"] for m in r.json().get("models", [])]
    print(f"✅ Ollama running | available models: {models}")
except Exception:
    print("❌ Ollama not reachable — run: ollama serve")

llm          = ChatOllama(model=MODEL, temperature=0)     # deterministic: critic, planner, evaluator
llm_creative = ChatOllama(model=MODEL, temperature=0.7)   # diverse: generator, idea generation


def ask(system: str, user: str, model=llm) -> str:
    """One LLM call with a system prompt and a user message."""
    return model.invoke([SystemMessage(system), HumanMessage(user)]).content


def show(title: str, text: str, width: int = 78) -> None:
    print(f"\n┌─ {title} " + "─" * max(0, width - len(title) - 4))
    for line in str(text).splitlines() or [""]:
        print("│ " + "\n│ ".join(textwrap.wrap(line, width - 2) or [""]))
    print("└" + "─" * (width - 1))


print(f"✅ LLM ready: {MODEL}")

✅ Ollama running | available models: ['nomic-embed-text:latest', 'qwen2.5:3b', 'llama3.1:latest']
✅ LLM ready: qwen2.5:3b


---
## Section 1 — Reflection: generate → critique → revise

In ReAct, the agent improves its answer by **acting on the world** (tools).
In **Reflection**, the agent improves its answer by **criticising itself**:

```
  task ──► GENERATOR ──► draft ──► CRITIC ──► feedback
               ▲                                  │
               └──────────── REVISE ◄─────────────┘
                     (until APPROVED or max_rounds)
```

The **same LLM** plays two roles with two different system prompts.
This works because evaluating a text is easier than writing it — the critic sees the draft "from outside".

> 📚 Self-Refine (Madaan et al., 2023) · Reflexion (Shinn et al., 2023)

### 1.1 — Describe your internship

In [ ]:
# 👉 Describe YOUR 2A internship in 2–3 sentences (company, mission, main result)
MY_INTERNSHIP = """
2A internship at <company>, <city>, <dates>.
Mission: <what you worked on>.
Main result: <what you achieved, with a number if possible>.
"""

TASK = (
    "Write a LinkedIn post of at most 80 words announcing the end of this internship. "
    "Professional tone, no more than 2 emojis, end with a thank-you to the team.\n\n"
    f"Internship: {MY_INTERNSHIP}"
)
print(TASK)

### 1.2 — The three roles

In [ ]:
GENERATOR_SYS = "You are a professional copywriter. Follow every constraint of the task exactly. Return only the text."

CRITIC_SYS = """You are a demanding editor. Review the DRAFT against the TASK.
List at most 3 concrete problems (length, tone, missing facts, clichés, emojis).
If the draft fully respects the task, answer with exactly one word: APPROVED"""


def generate(task: str) -> str:
    return ask(GENERATOR_SYS, task, llm_creative)


def critique(task: str, draft: str) -> str:
    return ask(CRITIC_SYS, f"TASK:\n{task}\n\nDRAFT:\n{draft}")


def revise(task: str, draft: str, feedback: str) -> str:
    return ask(
        GENERATOR_SYS,
        f"TASK:\n{task}\n\nPREVIOUS DRAFT:\n{draft}\n\nEDITOR FEEDBACK:\n{feedback}\n\n"
        "Rewrite the draft so that every point of the feedback is fixed. Return only the new draft.",
        llm_creative,
    )


print("✅ generate / critique / revise ready")

### 1.3 — The reflection loop

In [ ]:
def reflection_loop(task: str, max_rounds: int = 3, verbose: bool = True):
    draft = generate(task)
    history = [draft]
    if verbose:
        show(f"DRAFT 0  ({len(draft.split())} words)", draft)

    for r in range(1, max_rounds + 1):
        feedback = critique(task, draft)
        if verbose:
            show(f"CRITIC — round {r}", feedback)
        if "APPROVED" in feedback.upper():
            break
        draft = revise(task, draft, feedback)
        history.append(draft)
        if verbose:
            show(f"DRAFT {r}  ({len(draft.split())} words)", draft)

    return draft, history


final_post, history = reflection_loop(TASK)

print("\nWord count per draft:", [len(d.split()) for d in history])

**Observe:**
- Did the word count move towards the 80-word limit?
- Did the critic ever say `APPROVED`? Small models are often either **too lenient** (approve immediately)
  or **never satisfied** (always find something). Both make the free-text stopping rule unreliable.

### 1.4 ✏️ Exercise — A structured critic with a score

Replace the free-text critic with a **structured output** (CM2): the critic returns a score and a list of issues,
and the loop stops when `score >= threshold`.

**Hint:** some checks are better done **by code** than by the LLM. The word count is one of them.

In [ ]:
class Critique(BaseModel):
    score: int = Field(description="Overall quality of the draft, from 1 (bad) to 10 (excellent)")
    issues: list[str] = Field(description="At most 3 concrete problems to fix. Empty list if none.")


structured_critic = llm.with_structured_output(Critique)


def reflection_loop_v2(task: str, threshold: int = 8, max_words: int = 80, max_rounds: int = 3):
    """Return (final_draft, list_of_scores)."""
    # ── YOUR CODE HERE ────────────────────────────────────────────────────────
    # 1. draft = generate(task)
    # 2. For each round:
    #      c = structured_critic.invoke([SystemMessage(...), HumanMessage(...)])
    #      n_words = len(draft.split())                       ← check done by code
    #      if n_words > max_words: add an issue yourself, e.g. f"Too long: {n_words} words"
    #      record c.score ; stop if c.score >= threshold and n_words <= max_words
    #      otherwise draft = revise(task, draft, "\n".join(issues))
    # 3. return draft, scores
    pass
    # ─────────────────────────────────────────────────────────────────────────


# final_v2, scores = reflection_loop_v2(TASK)
# print("Scores per round:", scores)
# show("FINAL POST", final_v2)

**Questions:**
1. Which checks should be done by **code** rather than by the LLM critic? Why?
   > *Your answer:*
2. How many LLM calls does a reflection loop with `r` rounds cost (worst case)?
   > *Your answer:*

---
## Section 2 — Planning: Plan-and-Execute and Tree of Thoughts

ReAct **interleaves** thinking and acting: it decides one step at a time.
Planning patterns **think ahead** before acting.

| Pattern | Idea | Strength | Weakness |
|---------|------|----------|----------|
| **ReAct** (TP1) | think → act → observe, one step at a time | adapts to each observation | myopic, can loop |
| **Plan-and-Execute** | write the whole plan first, then execute each step | global vision, fewer "lost" steps | plan can be wrong and is not revised |
| **Tree of Thoughts** | explore several reasoning branches, score them, keep the best | avoids committing to the first idea | many LLM calls |

### 2.1 — Plan-and-Execute

```
  goal ──► PLANNER ──► [step 1, step 2, step 3, ...]
                              │
                              ▼
                   EXECUTOR runs step 1 ──► result 1 ─┐
                   EXECUTOR runs step 2 (sees result 1) ──► result 2 ─┐
                   ...                                                 ▼
                                                                 final results
```

In [ ]:
class Plan(BaseModel):
    steps: list[str] = Field(description="3 to 5 short, concrete, ordered steps")


planner = llm.with_structured_output(Plan)

GOAL = "Prepare a 5-minute oral presentation of my 2A internship for a jury of engineering teachers."

plan = planner.invoke([
    SystemMessage("You are a planning assistant. Break the goal into 3 to 5 concrete, ordered steps. "
                  "Do NOT execute the steps."),
    HumanMessage(f"Goal: {GOAL}\nContext: {MY_INTERNSHIP}"),
])

print("📋 PLAN")
for i, step in enumerate(plan.steps, 1):
    print(f"  {i}. {step}")

In [ ]:
def execute_plan(goal: str, plan: Plan, context: str) -> list[str]:
    results = []
    for i, step in enumerate(plan.steps, 1):
        previous = "\n".join(f"Step {j} result: {r[:300]}" for j, r in enumerate(results, 1))
        out = ask(
            "You execute ONE step of a plan. Be concrete and concise (max 80 words).",
            f"Goal: {goal}\nContext: {context}\n\nResults so far:\n{previous or '(none)'}\n\n"
            f"Current step: {step}",
        )
        results.append(out)
        show(f"STEP {i} — {step[:50]}", out)
    return results


step_results = execute_plan(GOAL, plan, MY_INTERNSHIP)

> 💡 **Re-planning.** A more robust version adds a **re-planner** after each step:
> *"given what we learned, should the remaining steps change?"*. This is a loop with a decision —
> exactly what LangGraph (TP4) is designed to express.

### 2.2 — Tree of Thoughts (ToT)

```
                         problem
             ┌──────────────┼──────────────┐
          idea A         idea B         idea C        ← generate k thoughts   (breadth)
       feas 6 / imp 5  feas 8 / imp 9  feas 4 / imp 7 ← evaluate each thought
                            │
                       develop idea B                  ← expand the best branch (depth)
```

> 📚 Yao et al., *Tree of Thoughts: Deliberate Problem Solving with LLMs*, 2023.

The problem below links to your **project**: find a first AI-agent use case for the company of your internship.

In [ ]:
PROBLEM = (
    "Propose ONE first use case of an LLM-based AI agent for the company where I did my internship. "
    f"Internship context: {MY_INTERNSHIP}"
)


class Evaluation(BaseModel):
    feasibility: int = Field(description="Feasibility in 3 months by 2 engineering students, from 1 to 10")
    impact: int = Field(description="Business impact for the company, from 1 to 10")
    justification: str = Field(description="One sentence justifying the two scores")


evaluator = llm.with_structured_output(Evaluation)


def generate_thoughts(problem: str, k: int = 3) -> list[str]:
    """Generate k ideas sequentially, asking each time for an idea different from the previous ones."""
    thoughts = []
    for _ in range(k):
        already = "\n".join(f"- {t}" for t in thoughts) or "(none)"
        idea = ask(
            "Propose ONE idea in at most 2 sentences. It must be clearly different from the ideas already listed.",
            f"Problem: {problem}\n\nIdeas already proposed:\n{already}",
            llm_creative,
        )
        thoughts.append(idea.strip())
    return thoughts


def evaluate_thought(problem: str, thought: str) -> Evaluation:
    return evaluator.invoke([
        SystemMessage("You are a strict technical reviewer. Score the idea honestly."),
        HumanMessage(f"Problem: {problem}\n\nIdea: {thought}"),
    ])


def tree_of_thoughts(problem: str, k: int = 3):
    thoughts = generate_thoughts(problem, k)
    scored = []
    for i, t in enumerate(thoughts, 1):
        e = evaluate_thought(problem, t)
        total = e.feasibility + e.impact
        scored.append((total, t, e))
        show(f"IDEA {i} — feasibility {e.feasibility} · impact {e.impact} · total {total}",
             f"{t}\n\n→ {e.justification}")

    best_total, best_idea, _ = max(scored, key=lambda x: x[0])
    development = ask(
        "Develop this idea into a short action plan: 4 bullet points (data, tools, agent pattern, success metric).",
        f"Problem: {problem}\n\nSelected idea: {best_idea}",
    )
    show(f"🏆 BEST IDEA (total {best_total}) — DEVELOPED", development)
    return best_idea, development


best_idea, development = tree_of_thoughts(PROBLEM, k=3)

### 2.3 ✏️ Questions

1. Count the LLM calls made by `tree_of_thoughts` with `k=3`. Compare with ReAct and Reflection.
   > *Your answer:*
2. Our ToT has depth 1 (we develop only the best branch). How would you implement **depth 2**
   (develop the 2 best ideas, score again, keep the best)? Describe it in 3 lines.
   > *Your answer:*
3. For which of these tasks would you choose ReAct, Reflection, Plan-and-Execute or ToT?
   *(a) answer a factual question from a database · (b) write a cover letter · (c) organise a 3-day event · (d) choose a system architecture*
   > *Your answer:*

---
## Section 3 — Multi-agent systems with CrewAI

In Sections 1–2, **one** LLM played several roles inside **your** Python loop.
A **multi-agent framework** makes those roles first-class objects that collaborate.

| Concept | CrewAI object | Key fields | Question it answers |
|---------|---------------|-----------|---------------------|
| Agent | `Agent` | `role`, `goal`, `backstory`, `tools`, `llm` | **Who** does the work? |
| Task | `Task` | `description`, `expected_output`, `agent`, `context` | **What** must be produced? |
| Crew | `Crew` | `agents`, `tasks`, `process` | **How** do they collaborate? |
| Process | `Process.sequential` / `Process.hierarchical` | — | in order / via a manager agent |

**Why `role`, `goal` and `backstory`?** CrewAI injects them into the agent's system prompt.
This *persona conditioning* steers vocabulary, level of detail and behaviour (e.g. "you never invent facts").
Internally, each agent still runs a **ReAct loop** with its tools — exactly what you coded in TP1.

```
   inputs = {"topic": ...}
          │
          ▼
   ┌──────────────┐     search_report(query)     ┌──────────────────┐
   │  RESEARCHER  │ ───────────────────────────► │ ChromaDB (TP2)   │
   │  ReAct loop  │ ◄─────────────────────────── │ your 2A report   │
   └──────┬───────┘          passages            └──────────────────┘
          │  research notes  (Task.context)
          ▼
   ┌──────────────┐
   │    WRITER    │ ──► internship_summary.md
   └──────────────┘
```

### 3.1 — Load the knowledge base from TP2

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

embeddings = OllamaEmbeddings(model=EMBED_MODEL)

if Path(CHROMA_DIR).exists():
    vectorstore = Chroma(
        collection_name="rapport_recursive",
        embedding_function=embeddings,
        persist_directory=CHROMA_DIR,
    )
    print(f"✅ TP2 index loaded — {vectorstore._collection.count()} chunks")
else:
    print("⚠️  chroma_db/ not found — building a small demo index (copy your TP2 folder here for the real thing)")
    SAMPLE = [
        ("Introduction. The internship took place at a mid-size industrial company. "
         "The objective was to automate the classification of customer support tickets.", 0),
        ("Methods. We compared a TF-IDF + logistic regression baseline with a fine-tuned BERT model. "
         "Class imbalance was handled with SMOTE.", 3),
        ("Results. The BERT model reached 87% accuracy versus 74% for the baseline. "
         "Processing time per ticket dropped from 4 minutes to 10 seconds.", 7),
        ("Difficulties and perspectives. Labelled data was scarce. Future work: active learning "
         "and deployment as an internal web service.", 9),
    ]
    docs = [Document(page_content=t, metadata={"page": p}) for t, p in SAMPLE]
    vectorstore = Chroma.from_documents(docs, embeddings, collection_name="demo_report")
    print(f"✅ Demo index built — {len(docs)} chunks")

### 3.2 — A CrewAI tool over the report

> ⚠️ CrewAI has **its own** `@tool` decorator (`crewai.tools.tool`). It is **not** the LangChain one from TP1/TP2 —
> we import it as `crew_tool` to avoid confusion. Same principle though: **the docstring is the tool description** the LLM reads.

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool as crew_tool


@crew_tool("search_report")
def search_report(query: str) -> str:
    """Search the student's internship report and return the 4 most relevant passages with their page numbers.
    Use short, specific queries such as 'project objectives', 'methods used' or 'results accuracy'."""
    docs = vectorstore.similarity_search(query, k=4)
    if not docs:
        return "No passage found."
    return "\n\n".join(f"[page {d.metadata.get('page', '?')}] {d.page_content}" for d in docs)


# Test the tool directly, without any agent
print(search_report.run(query="main objectives of the project")[:400])

### 3.3 — Agents: role, goal, backstory

In [ ]:
crew_llm = LLM(model=f"ollama/{CREW_MODEL}", base_url="http://localhost:11434", temperature=0)

researcher = Agent(
    role="Internship Report Analyst",
    goal="Extract accurate, sourced facts about {topic} from the internship report",
    backstory=(
        "You are a meticulous research assistant at IMT Mines Alès. "
        "You never invent facts: every claim you report comes from the search_report tool "
        "and is followed by its page number."
    ),
    tools=[search_report],
    llm=crew_llm,
    allow_delegation=False,
    max_iter=5,          # safety limit of the internal ReAct loop (cf. TP1 max_steps)
    verbose=True,
)

writer = Agent(
    role="Technical Writer",
    goal="Turn research notes into a clear executive summary for a jury of engineers",
    backstory=(
        "You are a science communicator who writes for busy engineering managers. "
        "You use only the facts provided by the researcher and you never add numbers that are not in the notes."
    ),
    llm=crew_llm,
    allow_delegation=False,
    verbose=True,
)

print("✅ Agents ready:", researcher.role, "|", writer.role)

### 3.4 — Tasks and crew

`expected_output` is a **contract**: the more precise it is, the better small models follow it.
`context=[research_task]` passes the researcher's output to the writer.

In [ ]:
research_task = Task(
    description=(
        "Search the internship report for information about: {topic}.\n"
        "Make 2 or 3 searches with different queries (objectives, methods, results, difficulties)."
    ),
    expected_output=(
        "A bullet list of 6 to 10 facts, each followed by its page number, "
        "e.g. '- The model reached 87% accuracy (p. 7)'."
    ),
    agent=researcher,
)

writing_task = Task(
    description="Write an executive summary of the internship about {topic}, using ONLY the research notes.",
    expected_output=(
        "A markdown document: a title, then 3 short paragraphs (context, work done, results). Max 200 words."
    ),
    agent=writer,
    context=[research_task],
    output_file="internship_summary.md",
)

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, writing_task],
    process=Process.sequential,
    verbose=True,
)

print("✅ Crew ready")

### 3.5 — Run the crew

This takes 1–3 minutes on a laptop. Read the verbose logs: you will recognise the **Thought / Action / Observation**
structure of ReAct inside the researcher.

> ⚠️ **Jupyter:** use `await crew.kickoff_async(...)`. Jupyter already runs an event loop, and recent CrewAI versions
> refuse the synchronous `crew.kickoff()` inside it (*"invoked synchronously from within a running event loop"*).
> In a plain `.py` script, `crew.kickoff(...)` is fine.

In [ ]:
TOPIC = "the main project of the internship: objectives, methods and results"

result = await crew.kickoff_async(inputs={"topic": TOPIC})

show("FINAL OUTPUT (writer)", result.raw)

In [ ]:
# What did each agent produce?
for task_output in result.tasks_output:
    show(f"{task_output.agent}", task_output.raw[:800])

print("\nToken usage:", result.token_usage)
print("Summary saved to: internship_summary.md")

### 3.6 ✏️ Exercise A — Add a reviewer agent (reflection, multi-agent version)

Section 1's critic becomes an **agent**. Create a `reviewer` that compares the summary with the research notes,
**flags every claim not supported by the notes**, and returns a corrected version.

Steps:
1. Create `reviewer = Agent(...)` with a role, a goal and a backstory of your choice (no tools needed).
2. Create `review_task = Task(...)` with `context=[research_task, writing_task]` and a precise `expected_output`
   (e.g. *"A list of unsupported claims, then the corrected summary"*).
3. Build `crew_v2` with the 3 agents and 3 tasks, and run it.

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# reviewer = Agent(
#     role=...,
#     goal=...,
#     backstory=...,
#     llm=crew_llm,
#     allow_delegation=False,
#     verbose=True,
# )
# review_task = Task(
#     description=...,
#     expected_output=...,
#     agent=reviewer,
#     context=[research_task, writing_task],
#     output_file="internship_summary_reviewed.md",
# )
# crew_v2 = Crew(...)
# result_v2 = await crew_v2.kickoff_async(inputs={"topic": TOPIC})
# show("REVIEWED SUMMARY", result_v2.raw)
pass
# ─────────────────────────────────────────────────────────────────────────────

### 3.7 ✏️ Exercise B — Does the backstory matter?

Re-create the `writer` with a very different backstory, for example:
*"You are an enthusiastic marketing intern who loves superlatives and emojis."*
Run the crew again with the same topic and compare the two summaries.

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────────
# writer_marketing = Agent(role="Technical Writer", goal=writer.goal, backstory="...", llm=crew_llm, verbose=True)
# writing_task_marketing = Task(description=writing_task.description,
#                               expected_output=writing_task.expected_output,
#                               agent=writer_marketing, context=[research_task])
# crew_marketing = Crew(agents=[researcher, writer_marketing],
#                       tasks=[research_task, writing_task_marketing],
#                       process=Process.sequential)
# result_marketing = await crew_marketing.kickoff_async(inputs={"topic": TOPIC})
# show("MARKETING WRITER", result_marketing.raw)
pass
# ─────────────────────────────────────────────────────────────────────────────

### 3.8 ✏️ Questions

1. Compare the two summaries of Exercise B. What changed? Did the facts change, or only the style?
   > *Your answer:*
2. With `qwen2.5:3b`, what failures did you observe (tool not called, wrong format, invented page numbers, loops)?
   Which ones did `max_iter`, `expected_output` or the backstory help with?
   > *Your answer:*
3. CrewAI vs your hand-written loops of Sections 1–2: what did you **gain** and what did you **lose** in control?
   > *Your answer:*
4. 🎁 *Bonus:* with `Process.hierarchical` and `manager_llm=crew_llm`, a manager agent assigns tasks itself.
   Why is this risky with a 3B model?
   > *Your answer:*

---
## Checkpoint — What you have built

```
  Pattern               Implementation                       Status
  ───────────────────── ──────────────────────────────────── ──────
  Reflection            generate → critique → revise loop     ✅
                        structured critic (score + issues)    ✏️
  Planning              Plan-and-Execute                      ✅
                        Tree of Thoughts (k ideas → best)     ✅
  Multi-agent           CrewAI researcher + writer            ✅
                        + reviewer agent                      ✏️
```

### Next — TP4 (second hour)

CrewAI hides the control flow. In TP4 you take it back with **LangGraph**: an explicit graph of nodes and edges,
a persistent state, conditional routing, and a **human-in-the-loop** that can pause the agent before a risky action.